In [1]:
import os,sys
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
sys.path.append('/root/liubo/TravDiT')  # 添加项目根目录到 Python 路径
from args import make_args
from util import load_raw_data,load_vec,generate_trajectory_prompt_temp,temporal_pattern_one_hot_batch
import random
from tqdm import tqdm 
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
args = make_args()
unique_poi_types = args.unique_poi_types
random.seed(args.seed)

task = args.task

In [2]:
from dataset.custom_dataset import CustomDataset
from torch.utils.data import DataLoader

all_data = []
dow_map = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
city_list = ['Changsha','Guangzhou', 'Shenzhen']


for city_id, city in enumerate(city_list):
    raw_data = load_raw_data(city=city)
    vec = load_vec(city=city).to(device)
    vec = vec[:, 2:2+args.poi_dim+args.pos_dim]
    sample_size = 10000
    # if city == 'Changsha':sample_size = 5000
    # else:sample_size = 10000

    indices = random.sample(range(len(raw_data)), sample_size)

    for i in indices:
        item = raw_data[i]
        item['city_id'] = city_id   # ✅ 添加 city_id 字段
        vec_seq = torch.stack([vec[int(rid)] for rid in item['traj_region_id']])
        all_data.append((item, vec_seq))

# 原始数据先传入 CustomDataset
random.shuffle(all_data)
all_raw_data, all_vec_seq = zip(*all_data)
all_raw_data = list(all_raw_data)
all_vec_seq = list(all_vec_seq)

full_dataset = CustomDataset(all_raw_data, all_vec_seq)
# 再拆分 Subset
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])
# 用 DataLoader 加载 Subset
train_dataloader = DataLoader(train_dataset, batch_size=args.Encoder_batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=args.Encoder_batch_size, shuffle=False)

/root/liubo/TravDiT/util.py:127: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  vec = torch.tensor(vec, dtype=torch.float32)


[INFO] Generated spatial POI+pos vec: torch.Size([890, 20])
[INFO] Generated spatial POI+pos vec: torch.Size([890, 20])
[INFO] Generated spatial POI+pos vec: torch.Size([890, 20])


In [3]:
# 预训练Transformer encoder for DiT
from model.model_Encoder import ST_Encoder,ST_Decoder
from model.st_layers_config.args import parse_args

encoder_config = parse_args()
encoder = ST_Encoder(config = encoder_config,dim_in = args.poi_dim+args.pos_dim, dim_out = args.latent_dim)
decoder = ST_Decoder(latent_dim = args.latent_dim, vocab_size = args.vocab_size,city_num=3,city_emb_dim=16)
encoder.to(device)
decoder.to(device)

ST_Encoder /root/liubo/TravDiT/model/st_layers_config


ST_Decoder(
  (city_emb): Embedding(3, 16)
  (decoder): Sequential(
    (0): LayerNorm((144,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=144, out_features=128, bias=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=128, out_features=890, bias=True)
  )
)

In [4]:
from torch.optim import AdamW

optimizer = AdamW(list(encoder.parameters()) + list(decoder.parameters()), lr=1e-2, weight_decay=0.01)
total_steps = len(train_dataloader)*args.Encoder_epoch
warmup_steps = int(total_steps * 0.1)  # 10% of total steps for warmup

from transformers import get_linear_schedule_with_warmup, get_cosine_schedule_with_warmup
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)
criterion = torch.nn.CrossEntropyLoss()  # ignore non-masked labels，这里看的是label是否mask

In [5]:
from sklearn.metrics import accuracy_score

for epoch in range(args.Encoder_epoch):
    epoch_loss = 0
    dataloader_tqdm = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{args.Encoder_epoch}", leave=False)
    encoder.train()
    decoder.train()
    for batch in dataloader_tqdm:
        region_id = batch['region_seq'].to(device)  # [B, K]
        labels = batch['region_seq'].to(device)        # [B, K]
        city_id = batch['city_id_seq'].to(device)      # [B, ]
        vec = batch['vec_seq'].to(device)  # [B, K, poi_dim + pos_dim]

        labels = labels.reshape(-1, args.K).long()  # [B*K]
        city_id = city_id.unsqueeze(1).expand(-1, args.K)  # [B, K]

        latent = encoder(input=vec.permute(0, 2, 1).unsqueeze(2))  # -> [B, T, 1, C]
        output = decoder(latent,city_id)                                     # -> [B, T, vocab_size]

        loss = criterion(output.view(-1, output.size(-1)), labels.view(-1))
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        epoch_loss += loss.item()
        dataloader_tqdm.set_postfix(loss=loss.item())
        avg_loss = epoch_loss / len(train_dataloader)
    print(f"[Epoch {epoch+1}/{args.Encoder_epoch}] Average Loss: {avg_loss:.6f}")

    with torch.no_grad():
        encoder.eval()
        decoder.eval()
        epoch_loss = 0
        all_preds = []
        all_trues = []

        for batch in val_dataloader:
            region_id = batch['region_seq'].to(device)  # [B, K]
            labels = batch['region_seq'].to(device)        # [B, K]
            city_id = batch['city_id_seq'].to(device)      # [B, ]
            vec = batch['vec_seq'].to(device)  # [B, K, poi_dim + pos_dim]

            labels = labels.reshape(-1, args.K).long()
            city_id = city_id.unsqueeze(1).expand(-1, args.K)  # [B, K]

            latent = encoder(input=vec.permute(0, 2, 1).unsqueeze(2))  # -> [B, T, 1, C]
            output = decoder(latent,city_id)                             # [B, T, vocab_size]

            val_loss = criterion(output.view(-1, output.size(-1)), labels.view(-1))
            epoch_loss += val_loss.item()

            # 收集预测和标签
            y_true = labels.view(-1).cpu()
            y_pred = output.argmax(dim=-1).view(-1).cpu()
            all_trues.append(y_true)
            all_preds.append(y_pred)

        # 拼接所有预测和标签
        all_trues = torch.cat(all_trues)
        all_preds = torch.cat(all_preds)
        val_acc = accuracy_score(all_trues.numpy(), all_preds.numpy())

        print(f"[Epoch {epoch+1}] Val Loss: {epoch_loss/len(val_dataloader):.4f} | Val Accuracy: {val_acc:.4f}")


[Epoch 1/80] Average Loss: 6.711579
[Epoch 1] Val Loss: 6.4229 | Val Accuracy: 0.0130


[Epoch 2/80] Average Loss: 5.896479
[Epoch 2] Val Loss: 5.4389 | Val Accuracy: 0.0350


[Epoch 3/80] Average Loss: 5.117917
[Epoch 3] Val Loss: 4.8409 | Val Accuracy: 0.0552


[Epoch 4/80] Average Loss: 4.738746
[Epoch 4] Val Loss: 4.4764 | Val Accuracy: 0.0762


[Epoch 5/80] Average Loss: 4.431481
[Epoch 5] Val Loss: 4.3494 | Val Accuracy: 0.0837


...

[Epoch 76/80] Average Loss: 1.109215
[Epoch 76] Val Loss: 1.2731 | Val Accuracy: 0.7458


[Epoch 77/80] Average Loss: 1.101943
[Epoch 77] Val Loss: 1.2707 | Val Accuracy: 0.7469


[Epoch 78/80] Average Loss: 1.095412
[Epoch 78] Val Loss: 1.2650 | Val Accuracy: 0.7481


[Epoch 79/80] Average Loss: 1.088323
[Epoch 79] Val Loss: 1.2609 | Val Accuracy: 0.7490


[Epoch 80/80] Average Loss: 1.081167
[Epoch 80] Val Loss: 1.2596 | Val Accuracy: 0.7497


In [6]:
# 保存encoder和decoder
torch.save(encoder.state_dict(), f"/root/liubo/TravDiT/{args.checkpt_path}/tuning/encoder/encoder_{task}_full0.25.pth")
torch.save(decoder.state_dict(), f"/root/liubo/TravDiT/{args.checkpt_path}/tuning/encoder/decoder_{task}_full0.25.pth")